In [1]:
from pathlib import Path
import datetime as dt

import numpy as np
import pandas as pd 

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.transforms import v2
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader, Subset


from custom_tools.tools import frame_video_split
from custom_tools.custom_logger import create_logger
from custom_tools.datasets import CADICADataset
from custom_tools.train_tools import befor_train_init, train_loop, compute_metrics
from custom_tools.architecture.ResNet_18_backbone import ResNet_18_Backbone

**LCA**

Сплит на **уровне видео** (одно видео пациента идет или в трейн или в тест).

Кросс-валидация не выполнялась

Две стадии обучения:
1) Заменяем последний слой - обучаем с большим LR (1e-3)
2) Берем лучший checkpoint, размораживаем всю сеть и обучаем с маленьким LR (1e-5)

В принципе, после первого этапа обучения, почти получили метрики из [статьи](https://arxiv.org/pdf/2402.00570) для данного типа артерии (LCA), для данного размера пакета и данной архитектуры сети (ResNet-18)

---
Разморозить всю сеть сразу (как в статье)

Уменьшил class_weights

---

Из статьи:
* Accuracy: **0.712**
* Balanced Accuracy: **0.656**
* F1: **0.794**

Метрики после первого этапа:
* Accuracy: 
* Balanced Accuracy: 
* F1: 

Вторая часть не выполнялась

In [2]:
# ===================== PARAMETERS =====================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# set seed
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

PATH_SELECTED_VIDEO = Path("../../datasets/CADICA/selectedVideos")
frame_cadica = pd.read_csv("../data/cadica_dataset.csv")

TRAIN_SIZE_DATA = 0.8
BATCH_SIZE = 64
input_img_shape = (3,224,224)

class_weights = [1.7, 1]
weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

LR = 1e-3
MOMENTUM = 0.9
LAMBDA_LR = 1e-4

EPOCHS = 20
EARLY_STOPPING = 5
STEP_BATCH_LOG = 10

base_dir_for_model_save = "../models_checkpoints" # dir for models save

model_name = "resnet_18_backbone" # model train name
prefix_train = "classifier_layer_train_v2" # prefix train



# ============ comment for log ============
comment = f""" 
===== general parameters =====
SEED: {SEED}
DEVICE: {DEVICE}

PATH_SELECTED_VIDEO: {PATH_SELECTED_VIDEO}

TRAIN_SIZE_DATA: {TRAIN_SIZE_DATA}
BATCH_SIZE: {BATCH_SIZE}
input_img_shape: {input_img_shape}

===== train parameters =====
class_weights: {class_weights}
weights: {weights}

LR: {LR}
MOMENTUM: {MOMENTUM}
LAMBDA_LR: {LAMBDA_LR}

EPOCHS: {EPOCHS}
EARLY_STOPPING: {EARLY_STOPPING}
STEP_BATCH_LOG: {STEP_BATCH_LOG}

===== about model and save =====
base_dir_for_model_save: {base_dir_for_model_save}

model_name: {model_name}
prefix_train: {prefix_train}

===== comments =====
LCA
split video level
SGDM optimizer
use augmentation
use ImageNet stats for (mean, std) normalization 
"""

## Load and split frame

In [ ]:
# === разбиение на LCA и RCA === 
lca_cadica_frame = frame_cadica[frame_cadica['artery'] == 'LCA']
# rca_cadica_frame = frame_cadica[frame_cadica['artery'] == 'RCA']

# === split LCA and RCA frames for train/test ===
frame_lca_train, frame_lca_test = frame_video_split(lca_cadica_frame, "video_id", TRAIN_SIZE_DATA)
frame_lca_train, frame_lca_valid = frame_video_split(frame_lca_train, "video_id", TRAIN_SIZE_DATA)

# === для преобразований ===
train_transform = v2.Compose([
    v2.RandomAffine(
        degrees=25,                          
        translate=(25/512, 25/512),          
        scale=(0.8, 1.7),                    
        interpolation=InterpolationMode.BILINEAR,
        fill=0                               
    ),
    v2.Resize((224, 224)),               
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True), # Преобразование типа, scale=True (масштабирование значений от 0 до 1)
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Нормализация от ImageNet
])

test_transform = v2.Compose([
    v2.Resize((224, 224)),               
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
print(lca_cadica_frame.shape, frame_lca_train.shape, frame_lca_test.shape)

print(len(lca_cadica_frame["video_id"].unique()), len(rca_cadica_frame["video_id"].unique()))
print(
    lca_cadica_frame.groupby("label")["video_id"].count()
)
print(
    rca_cadica_frame.groupby("label")["video_id"].count()
)

print(set(frame_lca_train["video_id"].unique()).intersection(set(frame_lca_test['video_id'].unique())))

216 118
label
0    1003
1    2225
Name: video_id, dtype: int64
label
0     617
1    1460
Name: video_id, dtype: int64


In [4]:
# ==== weights calculate ====
# class_0 = frame_lca_train[frame_lca_train['label'] == 0].shape[0]
# class_1 = frame_lca_train[frame_lca_train['label'] == 1].shape[0]
# total = class_0 + class_1
# weight_0 = class_1 / class_0
# weight_1 = 1
# print(class_0, class_1, total)
# print(weight_0, weight_1)

In [ ]:
# === datasets create ===
# LCA
train_lca_dataset = CADICADataset(frame_lca_train, "image_path", "label", transform=train_transform)
valid_lca_dataset = CADICADataset(frame_lca_valid, "image_path", "label", transform=test_transform)
test_lca_dataset = CADICADataset(frame_lca_test, "image_path", "label", transform=test_transform)

In [6]:
# === dev set create ===
indices = np.random.choice(len(train_lca_dataset), size=64)
dev_dataset = Subset(train_lca_dataset, indices)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=1
)

In [5]:
# === create loaders ===
train_loader = DataLoader(
    train_lca_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=1
)
valid_loader = DataLoader(
    valid_lca_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=1
)
test_loader = DataLoader(
    test_lca_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=1
)

In [6]:
# model init
model_resnet = ResNet_18_Backbone(input_img_shape, out_classes=2, freeze_backbone=False).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.SGD(model_resnet.parameters(), lr=LR, weight_decay=LAMBDA_LR, momentum=MOMENTUM)
# optimizer = optim.RMSprop(model_resnet.parameters(), lr=LR, weight_decay=LAMBDA_LR)

In [7]:
total_params = sum(p.numel() for p in model_resnet.parameters())
trainable_params = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)

print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых:       {trainable_params:,}")

Всего параметров: 11,177,538
Обучаемых:       11,177,538


In [8]:
time_now = dt.datetime.now().strftime("%Y-%m-%d_%H_%M_%S")

path2save = befor_train_init(
    base_dir_for_model_save,
    model_name,
    time_now,
    prefix_train
)

# === logger create ===
train_logger = create_logger(path2save, "train_log.log")

# === comment write log ===
train_logger.info(comment)

2026-04-09 13:20:39,590 - INFO -  
===== general parameters =====
SEED: 42
DEVICE: cuda

PATH_SELECTED_VIDEO: ..\..\datasets\CADICA\selectedVideos

TRAIN_SIZE_DATA: 0.8
BATCH_SIZE: 64
input_img_shape: (3, 224, 224)

===== train parameters =====
class_weights: [1.7, 1]
weights: tensor([1.7000, 1.0000], device='cuda:0')

LR: 0.001
MOMENTUM: 0.9
LAMBDA_LR: 0.0001

===== about model and save =====
base_dir_for_model_save: ../models_checkpoints

model_name: resnet_18_backbone
prefix_train: classifier_layer_train_v2

===== comments =====
LCA
split video level
SGDM optimizer
use augmentation
use ImageNet stats for (mean, std) normalization 



In [9]:
train_history = train_loop(
    model_resnet, 
    loss_fn, 
    optimizer, 
    train_loader, 
    valid_loader, 
    path_to_model_save=path2save,
    step_batch_log = STEP_BATCH_LOG,
    early_stopping_epoch = EARLY_STOPPING,
    logger = train_logger,
    device = DEVICE, 
    epochs = EPOCHS,
    seed = SEED
)

train: 9it [00:09,  1.03it/s]2026-04-09 13:20:58,266 - INFO - epoch 1 | batch 10 | train_loss: 0.01068538
train: 19it [00:18,  1.09it/s]2026-04-09 13:21:07,591 - INFO - epoch 1 | batch 20 | train_loss: 0.01039058
train: 29it [00:28,  1.07it/s]2026-04-09 13:21:17,225 - INFO - epoch 1 | batch 30 | train_loss: 0.01005800
train: 33it [00:32,  1.01it/s]
valid: 8it [00:10,  1.26s/it]
Computing metrics: 100%|██████████| 8/8 [00:07<00:00,  1.05it/s]
2026-04-09 13:21:38,653 - INFO - ================= epoch end =================
2026-04-09 13:21:38,654 - INFO - epoch 1 | train_loss: 0.64109684 | valid_loss: 0.67219371 | valid balanc. acc.: 0.57863289
2026-04-09 13:21:38,655 - INFO - metrics:
{'accuracy': 0.6875, 'balanced_accuracy': 0.5786328920057231, 'f1': 0.7919463087248322, 'precision': 0.7603092783505154, 'recall': 0.8263305322128851}
2026-04-09 13:21:40,235 - INFO - save checkpoint: ../models_checkpoints/resnet_18_backbone/2026-04-09_13_20_39_classifier_layer_train_v2/checkpoint/best_model

In [ ]:
metrics = compute_metrics(model_resnet, test_loader, DEVICE)
print(metrics)

train_logger.info(f"\nTest metrics:\n{metrics}")

Computing metrics: 100%|██████████| 11/11 [00:13<00:00,  1.25s/it]

{'accuracy': 0.6062874251497006, 'balanced_accuracy': 0.5019463255580637, 'f1': 0.7346115035317861, 'precision': 0.6642335766423357, 'recall': 0.8216704288939052}
